In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)

from sklearn.model_selection import ParameterGrid

### 데이터 불러오기

In [2]:
# 파일 불러오기 
target = "Risk_Label"

# Load datasets
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

train shape: (2351, 9) (2351,)
valid shape: (1438, 9) (1438,)
test shape: (822, 9) (822,)


## ParameterGrid SVM 적합

In [3]:
# =========================
# Parameter search space
# linear kernel에서는 gamma가 의미 없으므로 따로 분리
# =========================
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__gamma": ["scale", "auto", 0.01, 0.1],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["poly"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__gamma": ["scale", "auto", 0.01, 0.1],
        "svm__class_weight": [None],
    },
]


def compute_metrics(y_true, y_pred):
    """
    Accuracy, F1, Recall, Precision, G-Mean 계산.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    g_mean = np.sqrt(sensitivity * specificity)

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "g_mean": g_mean,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "cm": cm,
    }


def make_threshold_candidates(scores):
    """
    SVM decision_function score 기준 threshold 후보 생성.
    score >= threshold 이면 high-risk(1)로 분류.

    validation score의 모든 unique 값을 후보로 사용해서
    G-Mean 최대 threshold를 더 정확하게 찾는다.
    """
    scores = np.asarray(scores, dtype=float)

    thresholds = np.unique(
        np.r_[
            np.unique(scores),
            0.0,
            scores.min() - 1e-12,
            scores.max() + 1e-12
        ]
    )

    return thresholds



GMEAN_WITHIN_RATIO = 0.99


def select_gmean99_precision(results):
    """
    선택 기준:
    1) validation G-Mean 최고값의 99% 이상인 후보만 유지
    2) 그 후보들 중 validation Precision 최대
    3) 동률이면 G-Mean → F1 → Recall → Specificity → Accuracy 순서
    """
    if not results:
        return None

    max_gmean = max(r["g_mean"] for r in results)
    eligible = [
        r for r in results
        if r["g_mean"] >= GMEAN_WITHIN_RATIO * max_gmean
    ]

    eligible = sorted(
        eligible,
        key=lambda r: (
            r["precision"],
            r["g_mean"],
            r["f1"],
            r["recall"],
            r["specificity"],
            r["accuracy"],
        ),
        reverse=True
    )

    return eligible[0]


### 지표

In [4]:
# =========================
# ParameterGrid + validation G-Mean 99% 이상 후보 중 Precision 우선 threshold search
# =========================
search_history = []
best_result = None

all_params = list(ParameterGrid(param_grid))

print(f"Total parameter combinations: {len(all_params)}")
print("선택 기준: validation G-Mean 최고값의 99% 이상 후보 중 Precision 최대")

for i, params in enumerate(all_params, start=1):

    model = Pipeline(
        steps=[("svm", SVC())]
    )
    model.set_params(**params)
    model.fit(X_train, y_train)

    valid_score_tmp = model.decision_function(X_valid)
    threshold_candidates = make_threshold_candidates(valid_score_tmp)

    threshold_results = []

    for threshold in threshold_candidates:
        valid_pred_tmp = (valid_score_tmp >= threshold).astype(int)
        metrics = compute_metrics(y_valid, valid_pred_tmp)

        result = {
            "params": params,
            "threshold": float(threshold),
            **{k: v for k, v in metrics.items() if k != "cm"},
        }
        threshold_results.append(result)

    # 각 파라미터 조합 내부에서 먼저 G-Mean 99% 이상 후보 중 Precision 최고 cutoff 선택
    best_for_params = select_gmean99_precision(threshold_results)
    search_history.append(best_for_params)

    print(
        f"[{i:>3}/{len(all_params)}] "
        f"G-Mean={best_for_params['g_mean']:.4f}, "
        f"Precision={best_for_params['precision']:.4f}, "
        f"F1={best_for_params['f1']:.4f}, "
        f"Recall={best_for_params['recall']:.4f}, "
        f"Threshold={best_for_params['threshold']:.6f}, "
        f"Params={params}"
    )

# 전체 파라미터 조합 중에서도 G-Mean 최고값의 99% 이상 후보만 남기고 Precision 최고 선택
best_result = select_gmean99_precision(search_history)

# =========================
# Best model 재학습
# =========================
best_params = best_result["params"]
best_threshold = best_result["threshold"]
best_valid_gmean = best_result["g_mean"]

best_svm = Pipeline(
    steps=[("svm", SVC())]
)
best_svm.set_params(**best_params)
best_svm.fit(X_train, y_train)

# Validation prediction using selected threshold
valid_score = best_svm.decision_function(X_valid)
valid_pred = (valid_score >= best_threshold).astype(int)

valid_metrics = compute_metrics(y_valid, valid_pred)
cm_valid = valid_metrics["cm"]

valid_acc = valid_metrics["accuracy"]
valid_f1 = valid_metrics["f1"]
valid_recall = valid_metrics["recall"]
valid_precision = valid_metrics["precision"]
valid_g_mean = valid_metrics["g_mean"]

print("\n" + "=" * 70)
print("Best SVM Model selected by validation G-Mean 99% + Precision")
print("=" * 70)
print("Best Params      :", best_params)
print("Best Threshold   :", best_threshold)
print("Best Valid G-Mean:", best_valid_gmean)
print("Best Valid Precision:", best_result["precision"])

print("\n[Validation] Metrics")
print(f"Accuracy  : {valid_acc:.4f}")
print(f"F1-Score  : {valid_f1:.4f}")
print(f"Recall    : {valid_recall:.4f}")
print(f"Precision : {valid_precision:.4f}")
print(f"G-Mean    : {valid_g_mean:.4f}")
print(f"\n[Validation] Confusion Matrix:\n{cm_valid}")

print("\nClassification Report:")
print(classification_report(y_valid, valid_pred, digits=4, zero_division=0))

# Search history 확인 및 저장
search_df_raw = pd.DataFrame(search_history)
max_gmean_all = search_df_raw["g_mean"].max()
search_df = search_df_raw[
    search_df_raw["g_mean"] >= GMEAN_WITHIN_RATIO * max_gmean_all
].copy()

search_df = search_df.sort_values(
    by=["precision", "g_mean", "f1", "recall", "specificity", "accuracy"],
    ascending=[False, False, False, False, False, False]
).reset_index(drop=True)

result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)
search_save_path = result_dir / "02. SVM_GMean99_Precision_후보1.csv"
search_df.to_csv(search_save_path, index=False)
print(f"\n✓ 탐색 결과 저장 완료: {search_save_path}")

search_df.head()


Total parameter combinations: 45
선택 기준: validation G-Mean 최고값의 99% 이상 후보 중 Precision 최대
[  1/45] G-Mean=0.6721, Precision=0.2511, F1=0.3567, Recall=0.6154, Threshold=-0.996728, Params={'svm__C': 0.01, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  2/45] G-Mean=0.6732, Precision=0.2528, F1=0.3584, Recall=0.6154, Threshold=-0.966645, Params={'svm__C': 0.1, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  3/45] G-Mean=0.6700, Precision=0.2359, F1=0.3451, Recall=0.6429, Threshold=-0.913456, Params={'svm__C': 1, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  4/45] G-Mean=0.6666, Precision=0.2380, F1=0.3449, Recall=0.6264, Threshold=-0.877461, Params={'svm__C': 10, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  5/45] G-Mean=0.6691, Precision=0.2391, F1=0.3469, Recall=0.6319, Threshold=-0.880427, Params={'svm__C': 50, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  6/45] G-Mean=0.6772, Precision=0.2453, F1=0.3551, Recall=0.6429, Threshold=-0.997745, Params

,params,threshold,accuracy,f1,recall,precision,sensitivity,specificity,g_mean,tn,fp,fn,tp
0,"{'svm__C': 10, 'svm__class_weight': None, 'svm...",-0.999636,0.657163,0.348745,0.725275,0.229565,0.725275,0.647293,0.685175,813,443,50,132
1,"{'svm__C': 0.01, 'svm__class_weight': None, 's...",-0.999669,0.654381,0.343461,0.714286,0.226087,0.714286,0.645701,0.679128,811,445,52,130


### Test 성능 결과 

In [ ]:
# =========================
# Test evaluation using selected SVM model and selected threshold
# =========================
test_score = best_svm.decision_function(X_test)
test_pred = (test_score >= best_threshold).astype(int)

test_metrics = compute_metrics(y_test, test_pred)
cm_test = test_metrics["cm"]

test_acc = test_metrics["accuracy"]
test_f1 = test_metrics["f1"]
test_recall = test_metrics["recall"]
test_precision = test_metrics["precision"]
test_g_mean = test_metrics["g_mean"]

print("\n" + "=" * 70)
print("[Test] Metrics")
print("=" * 70)
print("Selected Threshold:", best_threshold)
print(f"Accuracy  : {test_acc:.4f}")
print(f"F1-Score  : {test_f1:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"G-Mean    : {test_g_mean:.4f}")
print(f"\n[Test] Confusion Matrix:\n{cm_test}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4, zero_division=0))


# =========================
# Save predictions
# =========================
result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

valid_pred_df = valid_df.copy()
valid_pred_df["SVM_score_gmean99_precision"] = valid_score
valid_pred_df["SVM_threshold_gmean99_precision"] = best_threshold
valid_pred_df["SVM_valid_pred_gmean99_precision"] = valid_pred

test_pred_df = test_df.copy()
test_pred_df["SVM_score_gmean99_precision"] = test_score
test_pred_df["SVM_threshold_gmean99_precision"] = best_threshold
test_pred_df["SVM_test_pred_gmean99_precision"] = test_pred

valid_save_path = result_dir / "02. SVM_valid_pred.csv"
test_save_path = result_dir / "02. SVM_test_pred.csv"

valid_pred_df.to_csv(valid_save_path, index=False)
test_pred_df.to_csv(test_save_path, index=False)

print(f"\n✓ 저장 완료: {valid_save_path}")
print(f"✓ 저장 완료: {test_save_path}")


[Test] Metrics
Selected Threshold: -0.9996361272630886
Accuracy  : 0.5657
F1-Score  : 0.2903
Recall    : 0.7374
Precision : 0.1807
G-Mean    : 0.6323

[Test] Confusion Matrix:
[[392 331]
 [ 26  73]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9378    0.5422    0.6871       723
           1     0.1807    0.7374    0.2903        99

    accuracy                         0.5657       822
   macro avg     0.5592    0.6398    0.4887       822
weighted avg     0.8466    0.5657    0.6393       822


✓ 저장 완료: ..\..\results\results_ML\02. SVM_valid_pred_gmean.csv
✓ 저장 완료: ..\..\results\results_ML\02. SVM_test_pred_gmean.csv


In [6]:
print("Best validation G-Mean:", best_valid_gmean)
print("Best validation Precision:", best_result["precision"])
print("Selected threshold:", best_threshold)
print("Test G-Mean:", test_g_mean)

print("\nBest Params:")
print(best_params)

print("\nValid Confusion Matrix:")
print(confusion_matrix(y_valid, valid_pred, labels=[0, 1]))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))

Best validation G-Mean: 0.6851753411555822
Best validation Precision: 0.22956521739130434
Selected threshold: -0.9996361272630886
Test G-Mean: 0.6322920446034003

Best Params:
{'svm__C': 10, 'svm__class_weight': None, 'svm__gamma': 0.01, 'svm__kernel': 'poly'}

Valid Confusion Matrix:
[[813 443]
 [ 50 132]]

Test Confusion Matrix:
[[392 331]
 [ 26  73]]
